In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the true state vector (target vector)
true_vec = np.array([0.6, 0.6, 0.6])  # Normalized
true_vec /= np.linalg.norm(true_vec)

# Function to rotate x and y basis vectors in the xy-plane
def rotated_basis(theta):
    x_rot = np.array([np.cos(theta), np.sin(theta), 0])
    y_rot = np.array([-np.sin(theta), np.cos(theta), 0])
    return x_rot, y_rot

# Projection of true_vec onto subspace spanned by rot_x and rot_y
def projection_onto_plane(true_vec, x_rot, y_rot):
    P = np.outer(x_rot, x_rot) + np.outer(y_rot, y_rot)
    return P @ true_vec

# Create frames for animation
frames = []
thetas = np.linspace(0, 2*np.pi, 60)

for theta in thetas:
    x_rot, y_rot = rotated_basis(theta)
    proj_vec = projection_onto_plane(true_vec, x_rot, y_rot)

    frame = go.Frame(
        data=[
            # Rotated x basis
            go.Scatter3d(x=[0, x_rot[0]], y=[0, x_rot[1]], z=[0, x_rot[2]],
                         mode='lines', line=dict(color='red', width=5), name='x_rot'),
            # Rotated y basis
            go.Scatter3d(x=[0, y_rot[0]], y=[0, y_rot[1]], z=[0, y_rot[2]],
                         mode='lines', line=dict(color='blue', width=5), name='y_rot'),
            # True state vector
            go.Scatter3d(x=[0, true_vec[0]], y=[0, true_vec[1]], z=[0, true_vec[2]],
                         mode='lines+markers', line=dict(color='black', width=4), name='True Vector'),
            # Projection vector
            go.Scatter3d(x=[0, proj_vec[0]], y=[0, proj_vec[1]], z=[0, proj_vec[2]],
                         mode='lines+markers', line=dict(color='green', width=4, dash='dash'), name='Projection')
        ],
        name=str(theta)
    )
    frames.append(frame)

# Initial plot
fig = go.Figure(
    data=frames[0].data,
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-1, 1]),
            yaxis=dict(range=[-1, 1]),
            zaxis=dict(range=[-1, 1]),
        ),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[dict(label='Play',
                          method='animate',
                          args=[None, dict(frame=dict(duration=100, redraw=True),
                                           fromcurrent=True, mode='immediate')])]
        )]
    ),
    frames=frames
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the true state vector (target vector)
true_vec = np.array([0.3, 0.5, 0.8])
true_vec = true_vec / np.linalg.norm(true_vec)

# Create a 2D subspace using spherical angles and in-plane rotation
def get_plane_basis(theta, phi, inplane_angle):
    # First basis vector (n̂) using spherical coords
    n = np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ])
    
    # Pick arbitrary orthogonal vector
    if abs(n[2]) < 0.9:
        temp = np.array([0, 0, 1])
    else:
        temp = np.array([1, 0, 0])
    
    u1 = np.cross(n, temp)
    u1 = u1 / np.linalg.norm(u1)
    
    # Second basis vector in plane
    u2 = np.cross(n, u1)
    
    # Rotate basis vectors within the plane
    x_rot = np.cos(inplane_angle) * u1 + np.sin(inplane_angle) * u2
    y_rot = np.cos(inplane_angle + np.pi/2) * u1 + np.sin(inplane_angle + np.pi/2) * u2
    
    return x_rot, y_rot

# Projection of true_vec onto 2D subspace
def projection_onto_plane(vec, b1, b2):
    P = np.outer(b1, b1) + np.outer(b2, b2)
    return P @ vec

# Animation setup
n_frames = 60
thetas = np.linspace(0.1, np.pi, n_frames)
phis = np.linspace(0, 2*np.pi, n_frames)
inplane_angles = np.linspace(0, 2*np.pi, n_frames)

frames = []

for t, p, a in zip(thetas, phis, inplane_angles):
    x_rot, y_rot = get_plane_basis(t, p, a)
    proj_vec = projection_onto_plane(true_vec, x_rot, y_rot)

    frame = go.Frame(data=[
        # Rotated x basis
        go.Scatter3d(x=[0, x_rot[0]], y=[0, x_rot[1]], z=[0, x_rot[2]],
                     mode='lines', line=dict(color='red', width=5), name='x_rot'),
        # Rotated y basis
        go.Scatter3d(x=[0, y_rot[0]], y=[0, y_rot[1]], z=[0, y_rot[2]],
                     mode='lines', line=dict(color='blue', width=5), name='y_rot'),
        # True state vector
        go.Scatter3d(x=[0, true_vec[0]], y=[0, true_vec[1]], z=[0, true_vec[2]],
                     mode='lines+markers', line=dict(color='black', width=4), name='True Vector'),
        # Projection vector
        go.Scatter3d(x=[0, proj_vec[0]], y=[0, proj_vec[1]], z=[0, proj_vec[2]],
                     mode='lines+markers', line=dict(color='green', width=4, dash='dash'), name='Projection')
    ])
    frames.append(frame)

# Initial frame
x_rot, y_rot = get_plane_basis(thetas[0], phis[0], inplane_angles[0])
proj_vec = projection_onto_plane(true_vec, x_rot, y_rot)

fig = go.Figure(
    data=[
        go.Scatter3d(x=[0, x_rot[0]], y=[0, x_rot[1]], z=[0, x_rot[2]],
                     mode='lines', line=dict(color='red', width=5), name='x_rot'),
        go.Scatter3d(x=[0, y_rot[0]], y=[0, y_rot[1]], z=[0, y_rot[2]],
                     mode='lines', line=dict(color='blue', width=5), name='y_rot'),
        go.Scatter3d(x=[0, true_vec[0]], y=[0, true_vec[1]], z=[0, true_vec[2]],
                     mode='lines+markers', line=dict(color='black', width=4), name='True Vector'),
        go.Scatter3d(x=[0, proj_vec[0]], y=[0, proj_vec[1]], z=[0, proj_vec[2]],
                     mode='lines+markers', line=dict(color='green', width=4, dash='dash'), name='Projection')
    ],
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-1, 1], title='x'),
            yaxis=dict(range=[-1, 1], title='y'),
            zaxis=dict(range=[-1, 1], title='z'),
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[dict(label='Play',
                          method='animate',
                          args=[None, dict(frame=dict(duration=100, redraw=True),
                                           fromcurrent=True, mode='immediate')])]
        )]
    ),
    frames=frames
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the true state vector (target vector)
true_vec = np.array([0.5, 0.5, 0.8])
true_vec = true_vec / np.linalg.norm(true_vec)


scene=dict(
    xaxis=dict(range=[-1, 1], title='x', backgroundcolor='black', color='white', gridcolor='gray'),
    yaxis=dict(range=[-1, 1], title='y', backgroundcolor='black', color='white', gridcolor='gray'),
    zaxis=dict(range=[-1, 1], title='z', backgroundcolor='black', color='white', gridcolor='gray'),
    aspectmode='cube'
),
paper_bgcolor='black',
plot_bgcolor='black',
font=dict(color='white'),



# Create a 2D subspace using spherical angles and in-plane rotation
def get_plane_basis(theta, phi, inplane_angle):
    # First basis vector (n̂) using spherical coords
    n = np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ])
    
    # Pick arbitrary orthogonal vector
    if abs(n[2]) < 0.9:
        temp = np.array([0, 0, 1])
    else:
        temp = np.array([1, 0, 0])
    
    u1 = np.cross(n, temp)
    u1 = u1 / np.linalg.norm(u1)
    
    # Second basis vector in plane
    u2 = np.cross(n, u1)
    
    # Rotate basis vectors within the plane
    x_rot = np.cos(inplane_angle) * u1 + np.sin(inplane_angle) * u2
    y_rot = np.cos(inplane_angle + np.pi/2) * u1 + np.sin(inplane_angle + np.pi/2) * u2
    
    return x_rot, y_rot

# Projection of true_vec onto 2D subspace
def projection_onto_plane(vec, b1, b2):
    P = np.outer(b1, b1) + np.outer(b2, b2)
    return P @ vec

# Animation setup
n_frames = 60

# Start with a plane that's far from optimal and gradually align it with the true vector
# We'll make the plane normal gradually align with the direction perpendicular to true_vec

# Find a direction perpendicular to true_vec for the final plane normal
perp_vec = np.array([1, 0, 0])
if abs(np.dot(true_vec, perp_vec)) > 0.8:
    perp_vec = np.array([0, 1, 0])

# Make it truly perpendicular
perp_vec = perp_vec - np.dot(perp_vec, true_vec) * true_vec
perp_vec = perp_vec / np.linalg.norm(perp_vec)

# Start with a random orientation and gradually converge
start_normal = np.array([1, 1, 1])
start_normal = start_normal / np.linalg.norm(start_normal)

frames = []

for i in range(n_frames):
    # Progress from 0 to 1
    progress = i / (n_frames - 1)
    
    # Use exponential convergence for more realistic behavior
    alpha = 1 - np.exp(-4 * progress)  # Converges to ~0.98 by the end
    
    # Interpolate the plane normal from start to perpendicular to true_vec
    current_normal = (1 - alpha) * start_normal + alpha * perp_vec
    current_normal = current_normal / np.linalg.norm(current_normal)
    
    # Convert to spherical coordinates for basis generation
    theta = np.arccos(np.clip(current_normal[2], -1, 1))
    phi = np.arctan2(current_normal[1], current_normal[0])
    
    # Add some rotation within the plane that also converges
    inplane_angle = (1 - alpha) * 2 * np.pi + alpha * 0
    
    x_rot, y_rot = get_plane_basis(theta, phi, inplane_angle)
    proj_vec = projection_onto_plane(true_vec, x_rot, y_rot)
    
    # Calculate projection error for display
    error = np.linalg.norm(true_vec - proj_vec)
    
    frame = go.Frame(data=[
        # Rotated x basis
        go.Scatter3d(x=[0, x_rot[0]], y=[0, x_rot[1]], z=[0, x_rot[2]],
                     mode='lines', line=dict(color='red', width=5), name='x basis'),
        # Rotated y basis
        go.Scatter3d(x=[0, y_rot[0]], y=[0, y_rot[1]], z=[0, y_rot[2]],
                     mode='lines', line=dict(color='blue', width=5), name='y basis'),
        # True state vector
        go.Scatter3d(x=[0, true_vec[0]], y=[0, true_vec[1]], z=[0, true_vec[2]],
                     mode='lines+markers', line=dict(color='black', width=4), 
                     marker=dict(size=8, symbol='diamond'), name='True Vector'),
        # Projection vector
        go.Scatter3d(x=[0, proj_vec[0]], y=[0, proj_vec[1]], z=[0, proj_vec[2]],
                     mode='lines+markers', line=dict(color='green', width=4, dash='dash'), 
                     marker=dict(size=6), name=f'Projection (error: {error:.3f})'),
        # Error vector (difference between true and projection)
        go.Scatter3d(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]], z=[proj_vec[2], true_vec[2]],
                     mode='lines', line=dict(color='red', width=2, dash='dot'), name='Error'),
    ])
    frames.append(frame)

# Initial frame
i = 0
progress = i / (n_frames - 1)
alpha = 1 - np.exp(-4 * progress)
current_normal = (1 - alpha) * start_normal + alpha * perp_vec
current_normal = current_normal / np.linalg.norm(current_normal)
theta = np.arccos(np.clip(current_normal[2], -1, 1))
phi = np.arctan2(current_normal[1], current_normal[0])
inplane_angle = (1 - alpha) * 2 * np.pi + alpha * 0

x_rot, y_rot = get_plane_basis(theta, phi, inplane_angle)
proj_vec = projection_onto_plane(true_vec, x_rot, y_rot)
error = np.linalg.norm(true_vec - proj_vec)

fig = go.Figure(
    data=[
        go.Scatter3d(x=[0, x_rot[0]], y=[0, x_rot[1]], z=[0, x_rot[2]],
                     mode='lines', line=dict(color='red', width=5), name='x basis'),
        go.Scatter3d(x=[0, y_rot[0]], y=[0, y_rot[1]], z=[0, y_rot[2]],
                     mode='lines', line=dict(color='blue', width=5), name='y basis'),
        go.Scatter3d(x=[0, true_vec[0]], y=[0, true_vec[1]], z=[0, true_vec[2]],
                     mode='lines+markers', line=dict(color='black', width=4), 
                     marker=dict(size=8, symbol='diamond'), name='True Vector'),
        go.Scatter3d(x=[0, proj_vec[0]], y=[0, proj_vec[1]], z=[0, proj_vec[2]],
                     mode='lines+markers', line=dict(color='green', width=4, dash='dash'), 
                     marker=dict(size=6), name=f'Projection (error: {error:.3f})'),
        go.Scatter3d(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]], z=[proj_vec[2], true_vec[2]],
                     mode='lines', line=dict(color='red', width=2, dash='dot'), name='Error'),
    ],
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-1, 1], title='x'),
            yaxis=dict(range=[-1, 1], title='y'),
            zaxis=dict(range=[-1, 1], title='z'),
            aspectmode='cube'
        ),
        title='Converging Projection: Plane Gradually Aligns with True Vector',
        margin=dict(l=0, r=0, b=0, t=40),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            y=1.02,
            x=0.1,
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=150, redraw=True),
                                      fromcurrent=True, mode='immediate')]),
                dict(label='⏸ Pause',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')])
            ]
        )]
    ),
    frames=frames
)

# Add annotation showing the convergence
fig.add_annotation(
    text="Watch the green projection vector converge to the black true vector<br>as the 2D subspace (red/blue plane) optimally aligns",
    xref="paper", yref="paper",
    x=0.02, y=0.98, xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Normalized target vector
vec = np.array([0.5, 0.5])
true_vec = vec / np.linalg.norm(vec)

n_frames = 60
frames = []

for i in range(n_frames):
    angle = i / (n_frames - 1) * np.pi / 4  # Rotate from 0 to 45 degrees

    # Rotated x and y basis vectors
    x_basis = np.array([np.cos(angle), np.sin(angle)])
    y_basis = np.array([-np.sin(angle), np.cos(angle)])

    # Project true vector onto x_basis
    proj_mag = np.dot(true_vec, x_basis)
    proj_vec = proj_mag * x_basis

    error_vec = true_vec - proj_vec
    error = np.linalg.norm(error_vec)

    frame = go.Frame(data=[
        # Rotated x basis
        go.Scatter(x=[0, x_basis[0]], y=[0, x_basis[1]],
                   mode='lines', line=dict(color='red', width=4), name='x basis'),

        # Rotated y basis
        go.Scatter(x=[0, y_basis[0]], y=[0, y_basis[1]],
                   mode='lines', line=dict(color='blue', width=4), name='y basis'),

        # True vector
        go.Scatter(x=[0, true_vec[0]], y=[0, true_vec[1]],
                   mode='lines+markers', line=dict(color='black', width=3),
                   marker=dict(size=8, symbol='diamond'), name='True Vector'),

        # Projection vector
        go.Scatter(x=[0, proj_vec[0]], y=[0, proj_vec[1]],
                   mode='lines+markers', line=dict(color='green', width=3, dash='dash'),
                   marker=dict(size=6), name=f'Projection (error: {error:.2f})'),

        # Error vector
        go.Scatter(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]],
                   mode='lines', line=dict(color='orange', width=2, dash='dot'), name='Error'),
    ])

    frames.append(frame)

# Initial plot setup
angle = 0
x_basis = np.array([np.cos(angle), np.sin(angle)])
y_basis = np.array([-np.sin(angle), np.cos(angle)])
proj_mag = np.dot(true_vec, x_basis)
proj_vec = proj_mag * x_basis
error_vec = true_vec - proj_vec
error = np.linalg.norm(error_vec)

fig = go.Figure(
    data=[
        go.Scatter(x=[0, x_basis[0]], y=[0, x_basis[1]],
                   mode='lines', line=dict(color='red', width=4), name='x basis'),
        go.Scatter(x=[0, y_basis[0]], y=[0, y_basis[1]],
                   mode='lines', line=dict(color='blue', width=4), name='y basis'),
        go.Scatter(x=[0, true_vec[0]], y=[0, true_vec[1]],
                   mode='lines+markers', line=dict(color='black', width=3),
                   marker=dict(size=8, symbol='diamond'), name='True Vector'),
        go.Scatter(x=[0, proj_vec[0]], y=[0, proj_vec[1]],
                   mode='lines+markers', line=dict(color='green', width=3, dash='dash'),
                   marker=dict(size=6), name=f'Projection (error: {error:.2f})'),
        go.Scatter(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]],
                   mode='lines', line=dict(color='orange', width=2, dash='dot'), name='Error'),
    ],
    layout=go.Layout(
        xaxis=dict(range=[-1, 1], zeroline=True, title="x"),
        yaxis=dict(range=[-0.2, 1.2], zeroline=True, title="y"),
        width=800,
        height=500,
        title="2D Rotating Basis: Projection and Error",
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=100, redraw=True),
                                      fromcurrent=True, mode='immediate')]),
                dict(label='⏸ Pause',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')])
            ]
        )]
    ),
    frames=frames
)

fig.add_annotation(
    text="Watch how the red (x) and blue (y) basis vectors rotate<br>to better align the x-basis with the target vector",
    xref="paper", yref="paper",
    x=0.02, y=0.98, xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="gray",
    borderwidth=1
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
import gif
import PIL.Image
import io



# Normalized target vector
vec = np.array([0.5, 0.5])
true_vec = vec / np.linalg.norm(vec)

n_frames = 60
frames = []
frames_pil = []

for i in range(n_frames):
    angle = i / (n_frames - 1) * np.pi / 4  # Rotate from 0 to 45 degrees

    # Rotated x and y basis vectors
    x_basis = np.array([np.cos(angle), np.sin(angle)])
    y_basis = np.array([-np.sin(angle), np.cos(angle)])

    # Project true vector onto x_basis
    proj_mag = np.dot(true_vec, x_basis)
    proj_vec = proj_mag * x_basis

    error_vec = true_vec - proj_vec
    error = np.linalg.norm(error_vec)

    frame = go.Frame(data=[
        # x basis vector
        go.Scatter(x=[0, x_basis[0]], y=[0, x_basis[1]],
                   mode='lines', line=dict(color='red', width=6), name='x basis'),

        # y basis vector
        go.Scatter(x=[0, y_basis[0]], y=[0, y_basis[1]],
                   mode='lines', line=dict(color='blue', width=6), name='y basis'),

        # True vector
        go.Scatter(x=[0, true_vec[0]], y=[0, true_vec[1]],
                   mode='lines+markers', line=dict(color='white', width=6),
                   marker=dict(size=10, symbol='diamond', color='white'), name='True Vector'),

        # Projection vector
        go.Scatter(x=[0, proj_vec[0]], y=[0, proj_vec[1]],
                   mode='lines+markers', line=dict(color='lime', width=4, dash='dash'),
                   marker=dict(size=8, color='lime'), name=f'Projection (error: {error:.2f})'),

        # Error vector
        go.Scatter(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]],
                   mode='lines', line=dict(color='orange', width=3, dash='dot'), name='Error'),
    ])
    frames.append(frame)

# Initial setup
angle = 0
x_basis = np.array([np.cos(angle), np.sin(angle)])
y_basis = np.array([-np.sin(angle), np.cos(angle)])
proj_mag = np.dot(true_vec, x_basis)
proj_vec = proj_mag * x_basis
error_vec = true_vec - proj_vec
error = np.linalg.norm(error_vec)

fig = go.Figure(
    data=[
        go.Scatter(x=[0, x_basis[0]], y=[0, x_basis[1]],
                   mode='lines', line=dict(color='red', width=6), name='x basis'),
        go.Scatter(x=[0, y_basis[0]], y=[0, y_basis[1]],
                   mode='lines', line=dict(color='blue', width=6), name='y basis'),
        go.Scatter(x=[0, true_vec[0]], y=[0, true_vec[1]],
                   mode='lines+markers', line=dict(color='white', width=6),
                   marker=dict(size=10, symbol='diamond', color='white'), name='True Vector'),
        go.Scatter(x=[0, proj_vec[0]], y=[0, proj_vec[1]],
                   mode='lines+markers', line=dict(color='lime', width=4, dash='dash'),
                   marker=dict(size=8, color='lime'), name=f'Projection (error: {error:.2f})'),
        go.Scatter(x=[proj_vec[0], true_vec[0]], y=[proj_vec[1], true_vec[1]],
                   mode='lines', line=dict(color='orange', width=3, dash='dot'), name='Error'),
    ],
    layout=go.Layout(
        xaxis=dict(range=[-1, 1], zeroline=True, title="x", color='white'),
        yaxis=dict(range=[-0.2, 1.2], zeroline=True, title="y", color='white'),
        width=900,
        height=600,
        paper_bgcolor='black',
        plot_bgcolor='black',
        font=dict(color='white'),
        #title="2D Rotating Basis: Projection and Error",
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=100, redraw=True),
                                      fromcurrent=True, mode='immediate')]),
                dict(label='⏸ Pause',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')])
            ]
        )]
    ),
    frames=frames
)

# fig.add_annotation(
#     xref="paper", yref="paper",
#     x=0.02, y=0.98, xanchor='left', yanchor='top',
#     showarrow=False,
#     font=dict(size=13, color='white'),
#     bgcolor="rgba(0,0,0,0.7)",
#     bordercolor="white",
#     borderwidth=1
# )


fig.show()

import plotly.io as pio
pio.write_html(fig, file='animation.html', auto_play=True)

# import os
# import PIL.Image
# import io

# # Ensure kaleido is installed or this will raise an error
# import plotly.io as pio

# # Generate frames
# gif_frames = []
# for frame in fig.frames:
#     fig.update(data=frame.data)
#     # Convert current frame to image
#     img_bytes = fig.to_image(format="png", engine="kaleido")
#     img = PIL.Image.open(io.BytesIO(img_bytes))
#     gif_frames.append(img)

# # Save GIF
# gif_frames[0].save("rotating_basis.gif",
#                    save_all=True,
#                    append_images=gif_frames[1:],
#                    optimize=True,
#                    duration=100,
#                    loop=0)

# print("✅ GIF saved as rotating_basis.gif")


